# Runtime Context

**Runtime context** is read-only data you pass into an agent at invocation time, things the agent should know about the current user or session but that aren't part of the conversation itself (for example, a user's preferences or ID).

In this notebook you'll:
1. Define a context schema with a dataclass and attach it to an agent.
2. Pass context in at invoke time.
3. Access that context from inside **tools** via `ToolRuntime`.

**Key learnings**
- Runtime context is read-only data passed into invoke
- Tools and middleware read context via ToolRuntime / ModelRequest
- Useful for user prefs, locale, and entitlements


### Load environment variables

`load_dotenv()` initializes the environment by loading the `.env` file so the model provider API key is available.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Define a context schema

We use a `@dataclass` to describe the shape of our runtime context. `ColourContext` holds two fields with defaults: `favourite_colour` and `least_favourite_colour`. This class acts as the schema the agent expects when context is passed in.

In [ ]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

### Create an agent with a context schema

We build an agent and register `context_schema=ColourContext`. This tells the agent to accept runtime context matching that dataclass. At this point the agent has no tools yet, it will rely on whatever context we supply at invoke time.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext  
)

### Invoke with context

We ask "What is my favourite colour?" and pass `context=ColourContext()` (using the default values). The context is available to the agent runtime during this call. Note that without a tool to read the context, the model can't reliably answer, that's what we fix in the next section.

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

### Inspect the response

`pprint` pretty-prints the response so we can read the message history and the agent's answer.

In [ ]:
from pprint import pprint

pprint(response)

## Accessing Context

To let the agent actually use the runtime context, we expose it through **tools**. A tool can declare a `ToolRuntime` parameter, and the framework injects the current runtime, giving the tool access to `runtime.context`.

### Define tools that read the context

Each tool takes a `runtime: ToolRuntime[ColourContext]` parameter. The framework injects the runtime automatically (the model doesn't supply it), so the tool can return values straight from `runtime.context`, here the user's favourite and least favourite colours.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

### Recreate the agent with the context-aware tools

We rebuild the agent, this time passing the two tools along with the same `context_schema`. Now the model can call `get_favourite_colour` / `get_least_favourite_colour`, which pull values from the runtime context.

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

### Invoke with default context

We ask the same question and pass the default `ColourContext()`. The agent calls the tool, which reads `favourite_colour` ("blue") from the context, and answers accordingly.

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

### Invoke with overridden context

Here we pass `ColourContext(favourite_colour="green")`, overriding the default. This demonstrates that the same agent produces a different answer purely based on the runtime context supplied at invoke time, without changing any code.

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)